# 09 - Detection Tuning Refinement

Bu notebook, Stage 09 tuning sonuçlarını daha dar parametre aralıklarıyla yeniden değerlendirir.

İlk bölüm büyük valid alt küme üzerinden threshold refinement yapar; ikinci bölüm kontrollü alt küme refinement akışını içerir.


## 1. Kütüphaneler

Bu bölümde tablo işlemleri, görüntü işleme, model yükleme ve feature çıkarma için kullanılan kütüphaneler yüklenir.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import math
import time
import traceback
import warnings

import numpy as np
import pandas as pd

import joblib
import cv2
import matplotlib.pyplot as plt

from skimage.feature import hog, local_binary_pattern

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)

print("Kütüphaneler yüklendi.")


## 2. Ayarlar

Bu bölümde threshold refinement modu, uzun çalışma korumaları ve IoU metrikleri tanımlanır.


In [ ]:
RANDOM_STATE = 42

# Default safe execution for this refinement notebook.
RUN_SMOKE_MODE = False
RUN_FULL_VALID_TUNING = True
RUN_QUALITATIVE_FIGURES = False

# Long-running output protection.
OVERWRITE_DETECTION_RESULTS = False
OVERWRITE_TABLES = False
OVERWRITE_FIGURES = False

DISPLAY_PREVIEW_ROWS = 10
MAX_CANDIDATES_PER_IMAGE = 500
FEATURE_EXTRACTION_BATCH_SIZE = 512

# Do not write a disk raw-score cache and do not share features across models.
# For each model and stride, raw scored windows are computed once and reused
# only for threshold / cluster / box-factor post-processing combinations.
USE_PER_MODEL_STRIDE_RAW_SCORE_REUSE = True

# Refinement mode identity.
TUNING_MODE_LABEL = "threshold_refinement_large_subset"

# Notebook 02 uses the same selected image lists from the completed
# large_valid_subset_700 run. If those CSVs cannot be found/read, it falls
# back to deterministic proportional sampling with the same target/random state.
USE_EXISTING_LARGE_VALID_SUBSET_SELECTED_IMAGES = True
EXISTING_LARGE_VALID_SUBSET_MODE_LABEL = "large_valid_subset_700"
USE_LARGE_VALID_SUBSET_FOR_TUNING = True
LARGE_VALID_SUBSET_TARGET_PER_DATASET = 700
LARGE_VALID_SUBSET_RANDOM_STATE = 42
LARGE_VALID_SUBSET_STRATEGY = "reuse_existing_selected_images_or_proportional_fallback"

# Controlled equal subset is not used in Notebook 02.
USE_CONTROLLED_EQUAL_SUBSET_FOR_TUNING = False
CONTROLLED_SUBSET_TARGETS = {}
CONTROLLED_SUBSET_RANDOM_STATE = 42

CLEAR_LARGE_VALID_SUBSET_CACHE_BEFORE_FULL_RUN = True

ENFORCE_OFFICIAL_24_MODEL_CONTRACT = True

POSTPROCESS_METHOD = "weighted_center"

MATCHING_IOU_THRESHOLDS = [0.10, 0.20, 0.30]
DEFAULT_PRIMARY_METRIC = "detection_f1_iou_0_2"
STRICT_COMPARISON_METRIC = "detection_f1_iou_0_3"

PHASE2_TOP_THRESHOLD_COUNT = 1

SMOKE_IMAGES_PER_DATASET = 5
SMOKE_MODELS_PER_GROUP = 1
SMOKE_USE_FULL_PHASE2_GRID = False

QUALITATIVE_EXAMPLES_PER_DATASET = 6

print("Ayarlar yüklendi.")


## 3. Dosya Yolları

Bu bölümde Stage 08 aday model tabloları, önceki büyük valid alt küme çıktıları ve 09 refinement çıktı klasörü tanımlanır.


In [ ]:
def auto_find_project_root(start_path=None):
    if start_path is None:
        start_path = Path.cwd()
    start_path = Path(start_path).resolve()
    candidates = [start_path] + list(start_path.parents)
    for candidate in candidates:
        has_repo_layout = (candidate / "approaches" / "traditional_ml").exists()
        has_data_dir = (candidate / "data").exists()
        if has_repo_layout and has_data_dir:
            return candidate
    return None


PROJECT_ROOT = auto_find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError("Proje kökü bulunamadı. Notebook'u repo içinde çalıştırın.")

APPROACH_ROOT = PROJECT_ROOT / "approaches" / "traditional_ml"
STAGE_NAME = "09_detection_tuning"
NOTEBOOK_NAME = "02_detection_tuning_threshold_refinement_large_subset"

OUTPUT_DIR = APPROACH_ROOT / "outputs" / STAGE_NAME / NOTEBOOK_NAME
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
INPUT_DIR = APPROACH_ROOT / "inputs" / STAGE_NAME

for folder in [OUTPUT_DIR, TABLES_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

STAGE08_SCREENING_TABLES_DIR = APPROACH_ROOT / "outputs" / "08_detection_screening" / "01_detection_screening_on_valid_subset" / "tables"
STAGE08_CALIBRATION_TABLES_DIR = APPROACH_ROOT / "outputs" / "08_detection_screening" / "00_preliminary_detection_parameter_calibration" / "tables"

CANDIDATE_MODELS_PATH = STAGE08_SCREENING_TABLES_DIR / "stage09_candidate_models.csv"
STANDARDIZED_PARAMS_PATH = STAGE08_SCREENING_TABLES_DIR / "standardized_detection_parameters.csv"
if not STANDARDIZED_PARAMS_PATH.exists():
    STANDARDIZED_PARAMS_PATH = STAGE08_CALIBRATION_TABLES_DIR / "standardized_detection_parameters.csv"

# Önceki büyük valid alt küme çıktıları bu refinement notebookunda tekrar kullanılabilir.
PREVIOUS_LARGE_SUBSET_TABLES_DIR_CANDIDATES = [
    INPUT_DIR,
]

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_FEATURES_DIR = PROJECT_ROOT / "data" / "features"
SELECTED_MODEL_POOL_DIR = PROJECT_ROOT / "outputs" / "models" / "selected_classification_model_pool"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CANDIDATE_MODELS_PATH:", CANDIDATE_MODELS_PATH)
print("STANDARDIZED_PARAMS_PATH:", STANDARDIZED_PARAMS_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("PREVIOUS_LARGE_SUBSET_TABLES_DIR_CANDIDATES:", PREVIOUS_LARGE_SUBSET_TABLES_DIR_CANDIDATES)


## 4. Refined Tuning Gridleri

Bu bölümde threshold refinement için daraltılmış parametre gridleri tanımlanır.


In [ ]:
LINEAR_SVM_THRESHOLD_GRID = {
    "dataset1_varroa": [1.4, 1.6, 1.8, 2.0, 2.2, 2.4, 2.6, 2.8, 3.0],
    "dataset2_honeybee": [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1],
}

RBF_SVM_THRESHOLD_GRID = {
    "dataset1_varroa": [0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5],
    "dataset2_honeybee": [0.7, 0.8, 0.9, 1.0, 1.1, 1.2],
}

RF_THRESHOLD_GRID = {
    "dataset1_varroa": [0.7, 0.8, 0.85, 0.9, 0.95, 0.99],
    "dataset2_honeybee": [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.99],
}

STRIDE_DIVISOR_GRID = {
    "dataset1_varroa": [4],
    "dataset2_honeybee": [4],
}

CLUSTER_IOU_THRESHOLD_GRID = [0.20, 0.25, 0.30]
WEIGHTED_BOX_SIZE_FACTOR_GRID = [1.00, 1.25]

ALGORITHM_TO_THRESHOLD_GRID = {
    "linear_svm": LINEAR_SVM_THRESHOLD_GRID,
    "rbf_svm": RBF_SVM_THRESHOLD_GRID,
    "random_forest": RF_THRESHOLD_GRID,
}

rows = []
for dataset_name in ["dataset1_varroa", "dataset2_honeybee"]:
    for algorithm_name, threshold_grid in ALGORITHM_TO_THRESHOLD_GRID.items():
        rows.append({
            "dataset_name": dataset_name,
            "algorithm_name": algorithm_name,
            "threshold_values": json.dumps(threshold_grid[dataset_name]),
            "threshold_count": len(threshold_grid[dataset_name]),
            "stride_divisor_values": json.dumps(STRIDE_DIVISOR_GRID[dataset_name]),
            "cluster_iou_threshold_values": json.dumps(CLUSTER_IOU_THRESHOLD_GRID),
            "weighted_box_size_factor_values": json.dumps(WEIGHTED_BOX_SIZE_FACTOR_GRID),
            "phase1_evaluations_per_model": len(threshold_grid[dataset_name]),
            "phase2_evaluations_per_model": PHASE2_TOP_THRESHOLD_COUNT * len(STRIDE_DIVISOR_GRID[dataset_name]) * len(CLUSTER_IOU_THRESHOLD_GRID) * len(WEIGHTED_BOX_SIZE_FACTOR_GRID),
            "tuning_mode_label": TUNING_MODE_LABEL,
            "use_existing_large_valid_subset_selected_images": USE_EXISTING_LARGE_VALID_SUBSET_SELECTED_IMAGES,
            "use_controlled_equal_subset_for_tuning": USE_CONTROLLED_EQUAL_SUBSET_FOR_TUNING,
            "controlled_subset_targets": json.dumps(CONTROLLED_SUBSET_TARGETS) if USE_CONTROLLED_EQUAL_SUBSET_FOR_TUNING else "",
            "smoke_use_full_phase2_grid": SMOKE_USE_FULL_PHASE2_GRID,
            "smoke_phase2_evaluations_per_model": (PHASE2_TOP_THRESHOLD_COUNT * len(STRIDE_DIVISOR_GRID[dataset_name]) * len(CLUSTER_IOU_THRESHOLD_GRID) * len(WEIGHTED_BOX_SIZE_FACTOR_GRID)) if SMOKE_USE_FULL_PHASE2_GRID else 1,
        })

tuning_grid_df = pd.DataFrame(rows)
print("Tuning grid hazır. Kayıt yardımcıları tanımlandıktan sonra gerektiğinde CSV olarak kaydedilecek.")
display(tuning_grid_df)


## 5. Veri Seti Ayarları

Bu bölümde valid görüntü, valid etiket ve PCA artifact klasörleri tanımlanır.


In [ ]:
DATASET_CONFIGS = {
    "dataset1_varroa": {
        "dataset_short": "dataset1",
        "valid_images_dir": DATA_RAW_DIR / "dataset1_varroa" / "valid" / "images",
        "valid_labels_dir": DATA_RAW_DIR / "dataset1_varroa" / "valid" / "labels",
        "pca_artifacts_dir": DATA_FEATURES_DIR / "dataset1_varroa" / "pca_artifacts",
        "positive_class_ids": {1},
    },
    "dataset2_honeybee": {
        "dataset_short": "dataset2",
        "valid_images_dir": DATA_RAW_DIR / "dataset2_honeybee" / "valid" / "images",
        "valid_labels_dir": DATA_RAW_DIR / "dataset2_honeybee" / "valid" / "labels",
        "pca_artifacts_dir": DATA_FEATURES_DIR / "dataset2_honeybee" / "pca_artifacts",
        "positive_class_ids": {0},
    },
}

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


## 6. Kayıt Yardımcıları

Bu bölümde CSV ve figür çıktıları için ortak kayıt fonksiyonları tanımlanır.


In [ ]:
SAVED_FILES = []


def relative_path(path):
    path = Path(path)

    try:
        return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except Exception:
        return path.as_posix()


def record_saved_file(path, file_type, description):
    path = Path(path)
    SAVED_FILES.append({
        "saved_at": datetime.now().isoformat(timespec="seconds"),
        "path": relative_path(path),
        "file_type": file_type,
        "description": description,
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else np.nan,
    })


def save_csv(df, path, description="", overwrite=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if overwrite is None:
        overwrite = OVERWRITE_TABLES

    if path.exists() and not overwrite:
        print(f"[SKIP] Mevcut CSV korundu: {relative_path(path)}")
        try:
            return pd.read_csv(path)
        except pd.errors.EmptyDataError:
            print(f"[INFO] Mevcut CSV boş olduğu için güncel tablo bellekte kullanılacak: {relative_path(path)}")
            return df

    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"[SAVE] CSV kaydedildi: {relative_path(path)} | shape={df.shape}")
    record_saved_file(path, "csv", description)
    return df


def save_current_figure(path, description="", overwrite=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if overwrite is None:
        overwrite = OVERWRITE_FIGURES

    if path.exists() and not overwrite:
        print(f"[SKIP] Mevcut figür korundu: {relative_path(path)}")
        return path

    plt.savefig(path, bbox_inches="tight", dpi=150)
    print(f"[SAVE] Figür kaydedildi: {relative_path(path)}")
    record_saved_file(path, "png", description)
    return path


def safe_divide(numerator, denominator):
    return float(numerator) / float(denominator) if denominator else 0.0


def metric_key(iou_threshold):
    return str(iou_threshold).replace("0.", "0_").replace(".", "_")


print("Yardımcı fonksiyonlar hazır.")


## 7. Girdi Kontrolü

Bu bölümde Stage 09 aday model listesi ve Stage 08 parametre tablosu okunup doğrulanır.


In [ ]:
required_input_paths = [CANDIDATE_MODELS_PATH, STANDARDIZED_PARAMS_PATH]
missing_inputs = [str(path) for path in required_input_paths if not Path(path).exists()]
if missing_inputs:
    raise FileNotFoundError("Missing required Stage 09 input files:\n" + "\n".join(missing_inputs))

candidate_models_df = pd.read_csv(CANDIDATE_MODELS_PATH)
standardized_params_df = pd.read_csv(STANDARDIZED_PARAMS_PATH)

required_candidate_columns = {
    "dataset_name", "algorithm_name", "model_file_name", "model_path",
    "patch_set", "patch_size", "feature_set", "threshold_column",
    "stage09_group_candidate_rank",
}
missing_candidate_columns = sorted(required_candidate_columns - set(candidate_models_df.columns))
if missing_candidate_columns:
    raise ValueError(f"stage09_candidate_models.csv is missing columns: {missing_candidate_columns}")

required_param_columns = {
    "dataset_name", "algorithm_name", "threshold_column", "threshold_value",
    "stride_divisor", "cluster_iou_threshold", "weighted_box_size_factor",
}
missing_param_columns = sorted(required_param_columns - set(standardized_params_df.columns))
if missing_param_columns:
    raise ValueError(f"standardized_detection_parameters.csv is missing columns: {missing_param_columns}")

candidate_models_df["model_path_original"] = candidate_models_df["model_path"]

def resolve_model_path(row):
    original = Path(str(row["model_path"]))
    if original.exists():
        return original
    return SELECTED_MODEL_POOL_DIR / row["dataset_name"] / row["algorithm_name"] / row["model_file_name"]

candidate_models_df["model_path_resolved"] = candidate_models_df.apply(resolve_model_path, axis=1).astype(str)
candidate_models_df["model_exists"] = candidate_models_df["model_path_resolved"].apply(lambda p: Path(p).exists())

group_counts = (
    candidate_models_df
    .groupby(["dataset_name", "algorithm_name"], as_index=False)
    .agg(model_count=("model_file_name", "count"))
    .sort_values(["dataset_name", "algorithm_name"])
)

validation_rows = [
    {"check_name": "candidate_model_count_is_24", "status": "PASS" if len(candidate_models_df) == 24 else "FAIL", "value": len(candidate_models_df)},
    {"check_name": "six_dataset_algorithm_groups", "status": "PASS" if len(group_counts) == 6 else "FAIL", "value": len(group_counts)},
    {"check_name": "each_group_has_four_models", "status": "PASS" if group_counts["model_count"].eq(4).all() else "FAIL", "value": json.dumps(group_counts.to_dict("records"))},
    {"check_name": "all_model_paths_exist", "status": "PASS" if candidate_models_df["model_exists"].all() else "WARN", "value": int(candidate_models_df["model_exists"].sum())},
]

for dataset_name, cfg in DATASET_CONFIGS.items():
    validation_rows.extend([
        {"check_name": f"{dataset_name}_valid_images_dir_exists", "status": "PASS" if cfg["valid_images_dir"].exists() else "FAIL", "value": str(cfg["valid_images_dir"])},
        {"check_name": f"{dataset_name}_valid_labels_dir_exists", "status": "PASS" if cfg["valid_labels_dir"].exists() else "FAIL", "value": str(cfg["valid_labels_dir"])},
        {"check_name": f"{dataset_name}_pca_artifacts_dir_exists", "status": "PASS" if cfg["pca_artifacts_dir"].exists() else "WARN", "value": str(cfg["pca_artifacts_dir"])},
    ])

candidate_validation_df = pd.DataFrame(validation_rows)
save_csv(candidate_validation_df, TABLES_DIR / "stage09_candidate_model_validation.csv", "Stage 09 candidate model validation checks")
save_csv(group_counts, TABLES_DIR / "stage09_candidate_group_counts.csv", "Candidate counts by dataset and algorithm")
save_csv(standardized_params_df, TABLES_DIR / "standardized_parameter_baseline.csv", "Stage 08 standardized baseline parameters copied for Stage 09")


if ENFORCE_OFFICIAL_24_MODEL_CONTRACT:
    hard_fail_checks = candidate_validation_df[
        (candidate_validation_df["status"] == "FAIL") &
        (candidate_validation_df["check_name"].isin([
            "candidate_model_count_is_24",
            "six_dataset_algorithm_groups",
            "each_group_has_four_models",
        ]))
    ]
    if not hard_fail_checks.empty:
        raise ValueError("Official Stage 09 candidate model contract failed:\n" + hard_fail_checks.to_string(index=False))
display(group_counts)
display(candidate_validation_df)


## 8. Valid Görüntü ve Ground Truth Yardımcıları

Bu bölümde valid split görüntü kayıtları ve YOLO etiketlerinden ground-truth kutuları hazırlanır.


In [ ]:
def find_image_files(images_dir):
    images_dir = Path(images_dir)
    return sorted([p for p in images_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS])

def parse_yolo_label_file(label_path, image_width, image_height, positive_class_ids):
    boxes = []
    invalid_rows = 0
    label_path = Path(label_path)
    if not label_path.exists():
        return boxes, invalid_rows

    for row_index, line in enumerate(label_path.read_text(encoding="utf-8", errors="ignore").splitlines()):
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 5:
            invalid_rows += 1
            continue
        try:
            class_id = int(float(parts[0]))
            x_center, y_center, width, height = map(float, parts[1:5])
        except Exception:
            invalid_rows += 1
            continue

        if class_id not in positive_class_ids:
            continue
        if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and width > 0 and height > 0):
            invalid_rows += 1
            continue

        x1 = max(0.0, min(float(image_width), (x_center - width / 2) * image_width))
        y1 = max(0.0, min(float(image_height), (y_center - height / 2) * image_height))
        x2 = max(0.0, min(float(image_width), (x_center + width / 2) * image_width))
        y2 = max(0.0, min(float(image_height), (y_center + height / 2) * image_height))
        if x2 <= x1 or y2 <= y1:
            invalid_rows += 1
            continue

        boxes.append({"class_id": class_id, "row_index": row_index, "x1": x1, "y1": y1, "x2": x2, "y2": y2})
    return boxes, invalid_rows

def build_valid_image_records(dataset_name):
    cfg = DATASET_CONFIGS[dataset_name]
    records = []
    for image_path in find_image_files(cfg["valid_images_dir"]):
        image = cv2.imread(str(image_path))
        label_path = cfg["valid_labels_dir"] / f"{image_path.stem}.txt"
        if image is None:
            records.append({
                "dataset_name": dataset_name, "image_path": relative_path(image_path), "label_path": relative_path(label_path),
                "image_id": image_path.stem, "image_width": np.nan, "image_height": np.nan,
                "gt_boxes": [], "gt_count": 0, "invalid_yolo_row_count": np.nan, "readable": False,
            })
            continue
        h, w = image.shape[:2]
        gt_boxes, invalid_rows = parse_yolo_label_file(label_path, w, h, cfg["positive_class_ids"])
        records.append({
            "dataset_name": dataset_name, "image_path": relative_path(image_path), "label_path": relative_path(label_path),
            "image_id": image_path.stem, "image_width": w, "image_height": h,
            "gt_boxes": gt_boxes, "gt_count": len(gt_boxes), "invalid_yolo_row_count": invalid_rows, "readable": True,
        })
    return pd.DataFrame(records)

valid_records_by_dataset = {dataset: build_valid_image_records(dataset) for dataset in DATASET_CONFIGS}

input_inventory = []
for dataset_name, df in valid_records_by_dataset.items():
    input_inventory.append({
        "dataset_name": dataset_name,
        "valid_image_count": len(df),
        "readable_image_count": int(df["readable"].sum()) if not df.empty else 0,
        "gt_object_count": int(df["gt_count"].sum()) if not df.empty else 0,
        "zero_varroa_image_count": int((df["gt_count"] == 0).sum()) if not df.empty else 0,
        "one_or_more_varroa_image_count": int((df["gt_count"] > 0).sum()) if not df.empty else 0,
        "invalid_yolo_row_count": int(pd.to_numeric(df["invalid_yolo_row_count"], errors="coerce").fillna(0).sum()) if not df.empty else 0,
    })

input_inventory_df = pd.DataFrame(input_inventory)
save_csv(input_inventory_df, TABLES_DIR / "input_inventory.csv", "Stage 09 valid split input inventory")
display(input_inventory_df)


## 9. Feature Çıkarma Yardımcıları

Bu bölümde HOG, HSV, LBP ve PCA feature çıkarma fonksiyonları tanımlanır.


In [ ]:
PCA_ARTIFACT_CACHE = {}

def parse_hog_variant(feature_set):
    if "hog8" in feature_set:
        return "hog8", 8
    if "hog16" in feature_set:
        return "hog16", 16
    return None, None

def is_pca_feature_set(feature_set):
    return "_pca" in feature_set

def uses_hog(feature_set):
    return "hog8" in feature_set or "hog16" in feature_set

def uses_hsv(feature_set):
    return "hsv" in feature_set

def uses_lbp(feature_set):
    return "lbp" in feature_set

def unwrap_model_artifact(artifact, artifact_path=None):
    """Return the estimator/pipeline object from either a raw estimator or a saved dict artifact."""
    if isinstance(artifact, dict):
        for key in ["model", "pipeline", "estimator", "best_estimator", "best_estimator_", "clf", "classifier"]:
            if key in artifact:
                artifact = artifact[key]
                break
    if not (hasattr(artifact, "decision_function") or hasattr(artifact, "predict_proba")):
        where = f" from {artifact_path}" if artifact_path is not None else ""
        raise TypeError(f"Loaded model artifact{where} has neither decision_function nor predict_proba.")
    return artifact

def load_model_artifact(model_path):
    artifact = joblib.load(model_path)
    return unwrap_model_artifact(artifact, artifact_path=model_path)

def unwrap_pca_artifact(artifact, pca_path=None):
    """Stage 03 PCA artifacts may be saved either directly or inside a dict."""
    if isinstance(artifact, dict):
        if "pca" in artifact:
            artifact = artifact["pca"]
        else:
            available = ", ".join(map(str, artifact.keys()))
            raise KeyError(f"PCA artifact dict does not contain key 'pca'. Available keys: {available}. Path: {pca_path}")
    if not hasattr(artifact, "transform"):
        raise TypeError(f"PCA artifact has no transform() method: {pca_path}")
    return artifact

def load_pca_artifact(dataset_name, patch_set, hog_variant):
    key = (dataset_name, patch_set, hog_variant)
    if key in PCA_ARTIFACT_CACHE:
        return PCA_ARTIFACT_CACHE[key]
    pca_path = DATASET_CONFIGS[dataset_name]["pca_artifacts_dir"] / f"{patch_set}__{hog_variant}_pca.joblib"
    if not pca_path.exists():
        raise FileNotFoundError(f"Missing PCA artifact: {pca_path}")
    artifact = joblib.load(pca_path)
    pca = unwrap_pca_artifact(artifact, pca_path=pca_path)
    PCA_ARTIFACT_CACHE[key] = pca
    return pca

def extract_hog_features(patch_bgr, pixels_per_cell):
    # cv2.imread returns BGR. BGR→GRAY here is equivalent to Stage 08's RGB→GRAY after RGB loading.
    # Keep uint8 grayscale input to match Stage 08 / Stage 03 HOG extraction behavior.
    gray = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2GRAY)
    return hog(
        gray, orientations=9, pixels_per_cell=(pixels_per_cell, pixels_per_cell),
        cells_per_block=(2, 2), block_norm="L2-Hys", transform_sqrt=True,
        feature_vector=True, channel_axis=None,
    ).astype(np.float32)

def normalized_hist(values, bins, value_range):
    hist, _ = np.histogram(values, bins=bins, range=value_range)
    hist = hist.astype(np.float32)
    total = hist.sum()
    if total > 0:
        hist /= total
    return hist

def extract_hsv_features(patch_bgr):
    # cv2.imread returns BGR; direct BGR→HSV is intentionally used and matches RGB→HSV after RGB conversion.
    hsv = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2HSV)
    h = normalized_hist(hsv[:, :, 0].ravel(), 16, (0, 180))
    s = normalized_hist(hsv[:, :, 1].ravel(), 8, (0, 256))
    v = normalized_hist(hsv[:, :, 2].ravel(), 8, (0, 256))
    return np.concatenate([h, s, v]).astype(np.float32)

def extract_lbp_features(patch_bgr):
    gray = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2GRAY)
    lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
    n_bins = 10  # P + 2 for P=8 uniform LBP.
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, n_bins + 1))
    hist = hist.astype(np.float32)
    total = hist.sum()
    if total > 0:
        hist /= total
    return hist

def extract_feature_matrix_from_patches(patches_bgr, dataset_name, patch_set, feature_set):
    """Extract a feature matrix for one batch of patches.

    PCA is applied in batch after raw HOG extraction. No cross-model or disk cache is used.
    """
    if not patches_bgr:
        return np.empty((0, 0), dtype=np.float32)

    matrices = []

    if uses_hog(feature_set):
        hog_variant, ppc = parse_hog_variant(feature_set)
        if hog_variant is None:
            raise ValueError(f"Unsupported HOG variant in feature_set: {feature_set}")
        raw_hog = np.vstack([extract_hog_features(patch, ppc) for patch in patches_bgr]).astype(np.float32)
        if is_pca_feature_set(feature_set):
            pca = load_pca_artifact(dataset_name, patch_set, hog_variant)
            hog_matrix = pca.transform(raw_hog).astype(np.float32)
        else:
            hog_matrix = raw_hog
        matrices.append(hog_matrix)

    if uses_hsv(feature_set):
        matrices.append(np.vstack([extract_hsv_features(patch) for patch in patches_bgr]).astype(np.float32))

    if uses_lbp(feature_set):
        matrices.append(np.vstack([extract_lbp_features(patch) for patch in patches_bgr]).astype(np.float32))

    if not matrices:
        raise ValueError(f"Unsupported feature_set: {feature_set}")

    return np.hstack(matrices).astype(np.float32)

def extract_feature_vector_from_patch(patch_bgr, dataset_name, patch_set, feature_set):
    """Single-patch wrapper retained for debugging/sanity checks."""
    return extract_feature_matrix_from_patches([patch_bgr], dataset_name, patch_set, feature_set)[0]


## 10. Detection Yardımcıları

Bu bölümde sliding-window aday üretimi, model skorlaması ve weighted-center post-process işlemleri tanımlanır.


In [ ]:
def generate_positions(length, patch_size, stride):
    if length < patch_size:
        return []
    positions = list(range(0, max(1, length - patch_size + 1), stride))
    last = length - patch_size
    if not positions or positions[-1] != last:
        positions.append(last)
    return sorted(set(int(p) for p in positions))

def generate_sliding_windows(image_width, image_height, patch_size, stride_divisor):
    # Stage 08 compatibility: keep a minimum stride of 4 px.
    stride = max(4, int(patch_size // stride_divisor))
    xs = generate_positions(image_width, patch_size, stride)
    ys = generate_positions(image_height, patch_size, stride)
    windows = []
    for y1 in ys:
        for x1 in xs:
            windows.append({
                "x1": float(x1),
                "y1": float(y1),
                "x2": float(x1 + patch_size),
                "y2": float(y1 + patch_size),
                "cx": float(x1 + patch_size / 2),
                "cy": float(y1 + patch_size / 2),
            })
    return windows, stride

def compute_iou(a, b):
    inter_x1, inter_y1 = max(a["x1"], b["x1"]), max(a["y1"], b["y1"])
    inter_x2, inter_y2 = min(a["x2"], b["x2"]), min(a["y2"], b["y2"])
    inter_w, inter_h = max(0.0, inter_x2 - inter_x1), max(0.0, inter_y2 - inter_y1)
    inter = inter_w * inter_h
    area_a = max(0.0, a["x2"] - a["x1"]) * max(0.0, a["y2"] - a["y1"])
    area_b = max(0.0, b["x2"] - b["x1"]) * max(0.0, b["y2"] - b["y1"])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def clip_box(box, image_width, image_height):
    box = dict(box)
    max_x1 = max(0.0, float(image_width) - 1.0)
    max_y1 = max(0.0, float(image_height) - 1.0)
    box["x1"] = max(0.0, min(max_x1, box["x1"]))
    box["y1"] = max(0.0, min(max_y1, box["y1"]))
    box["x2"] = max(0.0, min(float(image_width), box["x2"]))
    box["y2"] = max(0.0, min(float(image_height), box["y2"]))
    if box["x2"] < box["x1"]:
        box["x1"], box["x2"] = box["x2"], box["x1"]
    if box["y2"] < box["y1"]:
        box["y1"], box["y2"] = box["y2"], box["y1"]
    return box

def score_feature_matrix(model, X, threshold_column):
    if threshold_column == "predict_proba_score_threshold":
        if not hasattr(model, "predict_proba"):
            raise AttributeError("Model does not expose predict_proba required for predict_proba_score_threshold.")
        scores = model.predict_proba(X)[:, 1]
    else:
        if hasattr(model, "decision_function"):
            scores = model.decision_function(X)
        elif hasattr(model, "predict_proba"):
            scores = model.predict_proba(X)[:, 1]
        else:
            raise AttributeError("Model exposes neither decision_function nor predict_proba.")
    return np.asarray(scores).ravel()

def score_image_windows(model, image_bgr, dataset_name, patch_set, patch_size, feature_set, stride_divisor, threshold_column):
    h, w = image_bgr.shape[:2]
    windows, stride = generate_sliding_windows(w, h, patch_size, stride_divisor)

    scored = []
    batch_patches, batch_windows = [], []

    def flush_batch():
        nonlocal batch_patches, batch_windows, scored
        if not batch_patches:
            return
        X = extract_feature_matrix_from_patches(
            batch_patches,
            dataset_name=dataset_name,
            patch_set=patch_set,
            feature_set=feature_set,
        )
        scores = score_feature_matrix(model, X, threshold_column)
        scored.extend([{**win, "score": float(score)} for win, score in zip(batch_windows, scores)])
        batch_patches, batch_windows = [], []

    for win in windows:
        x1, y1, x2, y2 = map(int, [win["x1"], win["y1"], win["x2"], win["y2"]])
        patch = image_bgr[y1:y2, x1:x2]
        if patch.shape[:2] != (patch_size, patch_size):
            continue
        batch_patches.append(patch)
        batch_windows.append(win)

        if len(batch_patches) >= FEATURE_EXTRACTION_BATCH_SIZE:
            flush_batch()

    flush_batch()
    return scored, stride

def build_candidate_detections(scored_windows, threshold):
    candidates = [w for w in scored_windows if w["score"] >= threshold]
    candidates = sorted(candidates, key=lambda d: d["score"], reverse=True)
    return candidates[:MAX_CANDIDATES_PER_IMAGE]

def weighted_center_cluster(candidates, cluster_iou_threshold, weighted_box_size_factor, patch_size, threshold, image_width, image_height):
    remaining = sorted(candidates, key=lambda d: d["score"], reverse=True)
    final = []
    eps = 1e-6
    while remaining:
        seed = remaining[0]
        cluster, keep = [seed], []
        for det in remaining[1:]:
            if compute_iou(seed, det) >= cluster_iou_threshold:
                cluster.append(det)
            else:
                keep.append(det)

        scores = np.array([d["score"] for d in cluster], dtype=np.float64)
        weights = np.maximum(scores - float(threshold) + eps, eps)
        cx = float(np.sum(np.array([d["cx"] for d in cluster]) * weights) / np.sum(weights))
        cy = float(np.sum(np.array([d["cy"] for d in cluster]) * weights) / np.sum(weights))

        final_size = float(patch_size) * float(weighted_box_size_factor)
        box = {
            "x1": cx - final_size / 2,
            "y1": cy - final_size / 2,
            "x2": cx + final_size / 2,
            "y2": cy + final_size / 2,
            "cx": cx,
            "cy": cy,
            "score": float(np.max(scores)),
            "mean_score": float(np.mean(scores)),
            "cluster_size": int(len(cluster)),
        }
        final.append(clip_box(box, image_width, image_height))
        remaining = keep

    return sorted(final, key=lambda d: d["score"], reverse=True)

def run_detection_for_image(model, image_record, dataset_name, patch_set, patch_size, feature_set, threshold_column, threshold, stride_divisor, cluster_iou_threshold, weighted_box_size_factor):
    start_time = time.time()
    image_bgr = cv2.imread(str(image_record["image_path"]))
    if image_bgr is None:
        raise ValueError(f"Could not read image: {image_record['image_path']}")

    h, w = image_bgr.shape[:2]
    scored_windows, stride_px = score_image_windows(
        model=model,
        image_bgr=image_bgr,
        dataset_name=dataset_name,
        patch_set=patch_set,
        patch_size=patch_size,
        feature_set=feature_set,
        stride_divisor=stride_divisor,
        threshold_column=threshold_column,
    )
    candidates = build_candidate_detections(scored_windows, threshold)
    detections = weighted_center_cluster(
        candidates=candidates,
        cluster_iou_threshold=cluster_iou_threshold,
        weighted_box_size_factor=weighted_box_size_factor,
        patch_size=patch_size,
        threshold=threshold,
        image_width=w,
        image_height=h,
    )

    return {
        "image_id": image_record["image_id"],
        "gt_boxes": image_record["gt_boxes"],
        "detections": detections,
        "scored_window_count": len(scored_windows),
        "window_count": len(scored_windows),  # backward-compatible alias for quick inspection
        "candidate_count": len(candidates),
        "final_detection_count": len(detections),
        "stride_px": stride_px,
        "runtime_seconds": time.time() - start_time,
    }


## 11. Metrik Yardımcıları

Bu bölümde prediction-ground truth eşleştirme ve detection metrikleri hesaplanır.


In [ ]:
def match_detections_to_gt(detections, gt_boxes, iou_threshold):
    detections = sorted(detections, key=lambda d: d.get("score", 0.0), reverse=True)
    matched_gt = set()
    tp, fp = 0, 0
    matched_ious = []
    for det in detections:
        best_iou, best_idx = 0.0, None
        for idx, gt in enumerate(gt_boxes):
            if idx in matched_gt:
                continue
            iou = compute_iou(det, gt)
            if iou > best_iou:
                best_iou, best_idx = iou, idx
        if best_idx is not None and best_iou >= iou_threshold:
            tp += 1
            matched_gt.add(best_idx)
            matched_ious.append(best_iou)
        else:
            fp += 1
    return {"tp": tp, "fp": fp, "fn": len(gt_boxes) - len(matched_gt), "matched_ious": matched_ious}

def evaluate_detection_outputs(image_outputs, image_records=None, iou_thresholds=MATCHING_IOU_THRESHOLDS):
    """Evaluate per-image detection outputs.

    Ground-truth boxes are read directly from each output dict. The optional
    image_records argument is kept for backward compatibility with earlier
    calls but is not required.
    """
    total_images = len(image_outputs)
    total_window_count = sum(o.get("scored_window_count", o.get("window_count", 0)) for o in image_outputs)
    total_candidate_count = sum(o.get("candidate_count", 0) for o in image_outputs)
    total_final_detection_count = sum(o.get("final_detection_count", len(o.get("detections", []))) for o in image_outputs)
    total_runtime_seconds = sum(o.get("runtime_seconds", 0.0) for o in image_outputs)

    base = {
        "image_count": total_images,
        "total_window_count": total_window_count,
        "total_candidate_count": total_candidate_count,
        "total_final_detection_count": total_final_detection_count,
        "mean_candidates_per_image": safe_divide(total_candidate_count, total_images),
        "mean_final_detections_per_image": safe_divide(total_final_detection_count, total_images),
        "total_runtime_seconds": total_runtime_seconds,
        "mean_runtime_seconds_per_image": safe_divide(total_runtime_seconds, total_images),
    }

    for iou_threshold in iou_thresholds:
        key = metric_key(iou_threshold)
        tp_total = fp_total = fn_total = 0
        matched_ious_all = []
        negative_image_count = 0
        negative_total_fp = 0
        negative_images_with_fp = 0

        for output in image_outputs:
            gt_boxes = output.get("gt_boxes", [])
            detections = output.get("detections", [])
            match = match_detections_to_gt(detections, gt_boxes, iou_threshold)
            tp_total += match["tp"]
            fp_total += match["fp"]
            fn_total += match["fn"]
            matched_ious_all.extend(match["matched_ious"])

            if len(gt_boxes) == 0:
                negative_image_count += 1
                negative_total_fp += len(detections)
                if len(detections) > 0:
                    negative_images_with_fp += 1

        precision = safe_divide(tp_total, tp_total + fp_total)
        recall = safe_divide(tp_total, tp_total + fn_total)
        f1 = safe_divide(2 * precision * recall, precision + recall)

        base.update({
            f"tp_iou_{key}": tp_total,
            f"fp_iou_{key}": fp_total,
            f"fn_iou_{key}": fn_total,
            f"detection_precision_iou_{key}": precision,
            f"detection_recall_iou_{key}": recall,
            f"detection_f1_iou_{key}": f1,
            f"false_positives_per_image_iou_{key}": safe_divide(fp_total, total_images),
            f"false_negatives_per_image_iou_{key}": safe_divide(fn_total, total_images),
            f"negative_image_count_iou_{key}": negative_image_count,
            f"negative_total_fp_iou_{key}": negative_total_fp,
            f"negative_fp_per_negative_image_iou_{key}": safe_divide(negative_total_fp, negative_image_count),
            f"negative_accuracy_iou_{key}": safe_divide(negative_image_count - negative_images_with_fp, negative_image_count),
            f"mean_matched_iou_iou_{key}": float(np.mean(matched_ious_all)) if matched_ious_all else 0.0,
        })
    return base

def sort_results(df, primary_metric=DEFAULT_PRIMARY_METRIC, iou_key="0_2"):
    if df.empty:
        return df.copy()
    fp_col = f"false_positives_per_image_iou_{iou_key}"
    recall_col = f"detection_recall_iou_{iou_key}"
    sort_cols = [primary_metric, fp_col, recall_col, STRICT_COMPARISON_METRIC, "mean_final_detections_per_image"]
    ascending = [False, True, False, False, True]
    existing = [(c, a) for c, a in zip(sort_cols, ascending) if c in df.columns]
    return df.sort_values([c for c, _ in existing], ascending=[a for _, a in existing]).reset_index(drop=True) if existing else df.copy()


## 12. Tuning Çalıştırma Yardımcıları

Bu bölümde tek bir veri seti-algoritma grubu için refinement tuning akışı tanımlanır.


In [ ]:
def get_baseline_params(dataset_name, algorithm_name):
    subset = standardized_params_df[
        (standardized_params_df["dataset_name"] == dataset_name) &
        (standardized_params_df["algorithm_name"] == algorithm_name)
    ]
    if subset.empty:
        raise ValueError(f"No standardized baseline parameters for {dataset_name} / {algorithm_name}")
    row = subset.iloc[0].to_dict()
    return {
        "threshold_column": row["threshold_column"],
        "threshold_value": float(row["threshold_value"]),
        "stride_divisor": int(row["stride_divisor"]),
        "cluster_iou_threshold": float(row["cluster_iou_threshold"]),
        "weighted_box_size_factor": float(row["weighted_box_size_factor"]),
    }

def get_threshold_values(dataset_name, algorithm_name):
    return ALGORITHM_TO_THRESHOLD_GRID[algorithm_name][dataset_name]

def get_group_models(dataset_name, algorithm_name, smoke=False):
    group = candidate_models_df[
        (candidate_models_df["dataset_name"] == dataset_name) &
        (candidate_models_df["algorithm_name"] == algorithm_name)
    ].copy().sort_values("stage09_group_candidate_rank")
    return group.head(SMOKE_MODELS_PER_GROUP) if smoke else group

LARGE_VALID_SUBSET_CACHE = {}

if RUN_FULL_VALID_TUNING and USE_LARGE_VALID_SUBSET_FOR_TUNING and CLEAR_LARGE_VALID_SUBSET_CACHE_BEFORE_FULL_RUN:
    LARGE_VALID_SUBSET_CACHE.clear()
    print("Cleared subset cache before full tuning run.")

def get_run_mode_label(smoke=False):
    if smoke:
        return "smoke"
    return TUNING_MODE_LABEL

def add_gt_count_group(df):
    df = df.copy()

    def group_count(value):
        value = int(value)
        if value == 0:
            return "zero"
        if value == 1:
            return "one"
        if value == 2:
            return "two"
        return "three_plus"

    df["gt_count_group"] = df["gt_count"].apply(group_count)
    return df

def add_controlled_count_group(df):
    df = df.copy()

    def group_count(value):
        value = int(value)
        if value == 0:
            return "zero"
        if value == 1:
            return "one"
        return "two_or_more"

    df["controlled_gt_count_group"] = df["gt_count"].apply(group_count)
    return df

def select_smoke_records(df):
    positives = df[df["gt_count"] > 0].head(math.ceil(SMOKE_IMAGES_PER_DATASET / 2))
    negatives = df[df["gt_count"] == 0].head(SMOKE_IMAGES_PER_DATASET - len(positives))
    selected = pd.concat([positives, negatives], ignore_index=True)
    if len(selected) < SMOKE_IMAGES_PER_DATASET:
        used = set(selected["image_id"].astype(str))
        selected = pd.concat([
            selected,
            df[~df["image_id"].astype(str).isin(used)].head(SMOKE_IMAGES_PER_DATASET - len(selected)),
        ], ignore_index=True)
    return selected.head(SMOKE_IMAGES_PER_DATASET).reset_index(drop=True)

def select_proportional_stratified_subset(df, target_count, random_state):
    df = add_gt_count_group(df)

    if len(df) <= target_count:
        return df.sort_values("image_id").reset_index(drop=True)

    group_order = ["zero", "one", "two", "three_plus"]
    group_counts = df["gt_count_group"].value_counts().to_dict()

    allocations = {}
    fractional_parts = {}
    for group in group_order:
        available = int(group_counts.get(group, 0))
        exact = target_count * available / len(df)
        base = int(np.floor(exact))
        base = min(base, available)
        allocations[group] = base
        fractional_parts[group] = exact - base

    allocated_total = sum(allocations.values())
    remaining_slots = target_count - allocated_total

    while remaining_slots > 0:
        candidates = [
            group for group in group_order
            if allocations[group] < int(group_counts.get(group, 0))
        ]
        if not candidates:
            break
        best_group = max(candidates, key=lambda g: fractional_parts[g])
        allocations[best_group] += 1
        fractional_parts[best_group] = 0.0
        remaining_slots -= 1

    selected_parts = []
    for group in group_order:
        group_df = df[df["gt_count_group"] == group]
        n_group = int(allocations.get(group, 0))
        if n_group <= 0 or group_df.empty:
            continue
        selected_parts.append(group_df.sample(n=n_group, random_state=random_state))

    selected = pd.concat(selected_parts, ignore_index=True) if selected_parts else pd.DataFrame(columns=df.columns)

    if len(selected) < target_count:
        selected_ids = set(selected["image_id"].astype(str)) if not selected.empty else set()
        remaining_df = df[~df["image_id"].astype(str).isin(selected_ids)]
        need = target_count - len(selected)
        if not remaining_df.empty:
            selected = pd.concat([
                selected,
                remaining_df.sample(n=min(need, len(remaining_df)), random_state=random_state),
            ], ignore_index=True)

    if len(selected) > target_count:
        selected = selected.sample(n=target_count, random_state=random_state)

    return selected.sort_values("image_id").reset_index(drop=True)

def select_controlled_equal_subset(df, targets, random_state):
    df = add_controlled_count_group(df)
    selected_parts = []
    availability_rows = []

    for group, target_count in targets.items():
        group_df = df[df["controlled_gt_count_group"] == group]
        available = len(group_df)
        availability_rows.append({"controlled_gt_count_group": group, "available_count": available, "target_count": target_count})
        if available < target_count:
            raise ValueError(
                f"Controlled subset target cannot be satisfied for group={group}: "
                f"available={available}, target={target_count}"
            )
        selected_parts.append(group_df.sample(n=int(target_count), random_state=random_state))

    selected = pd.concat(selected_parts, ignore_index=True)
    return selected.sort_values("image_id").reset_index(drop=True), pd.DataFrame(availability_rows)

def load_existing_selected_images_for_dataset(dataset_name, full_df):
    expected_name = f"{EXISTING_LARGE_VALID_SUBSET_MODE_LABEL}__{dataset_name}__selected_images.csv"
    checked_paths = []

    for tables_dir in PREVIOUS_LARGE_SUBSET_TABLES_DIR_CANDIDATES:
        path = Path(tables_dir) / expected_name
        checked_paths.append(path)
        if not path.exists():
            continue
        try:
            selected_csv = pd.read_csv(path)
        except Exception as exc:
            print(f"Could not read existing selected-images file {path}: {exc}")
            continue

        if "image_id" not in selected_csv.columns:
            print(f"Existing selected-images file has no image_id column: {path}")
            continue

        selected_ids = set(selected_csv["image_id"].astype(str))
        selected = full_df[full_df["image_id"].astype(str).isin(selected_ids)].copy()
        if len(selected) != len(selected_ids):
            print(
                f"Existing selected-images file partially matched for {dataset_name}: "
                f"matched={len(selected)}, expected={len(selected_ids)}. Falling back."
            )
            continue

        print(f"Loaded existing selected images for {dataset_name}: {path} ({len(selected)} rows)")
        return selected.sort_values("image_id").reset_index(drop=True), path

    print("Existing selected-images file not found/readable. Checked:")
    for path in checked_paths:
        print(" -", path)
    return None, None

def summarize_subset_distribution(full_df, subset_df, dataset_name, mode_label):
    full_grouped = (
        add_gt_count_group(full_df)
        .groupby("gt_count_group", as_index=False)
        .agg(full_count=("image_id", "count"))
    )
    subset_grouped = (
        add_gt_count_group(subset_df)
        .groupby("gt_count_group", as_index=False)
        .agg(subset_count=("image_id", "count"))
    )
    summary = full_grouped.merge(subset_grouped, on="gt_count_group", how="outer").fillna(0)
    summary["dataset_name"] = dataset_name
    summary["mode_label"] = mode_label
    summary["full_count"] = summary["full_count"].astype(int)
    summary["subset_count"] = summary["subset_count"].astype(int)
    summary["full_total"] = int(len(full_df))
    summary["subset_total"] = int(len(subset_df))
    summary["full_ratio"] = summary["full_count"] / max(1, len(full_df))
    summary["subset_ratio"] = summary["subset_count"] / max(1, len(subset_df))
    summary["ratio_difference"] = summary["subset_ratio"] - summary["full_ratio"]
    return summary[[
        "dataset_name", "mode_label", "gt_count_group", "full_count", "subset_count",
        "full_total", "subset_total", "full_ratio", "subset_ratio", "ratio_difference",
    ]]

def summarize_controlled_subset_distribution(full_df, subset_df, dataset_name, mode_label):
    full_grouped = (
        add_controlled_count_group(full_df)
        .groupby("controlled_gt_count_group", as_index=False)
        .agg(full_count=("image_id", "count"))
    )
    subset_grouped = (
        add_controlled_count_group(subset_df)
        .groupby("controlled_gt_count_group", as_index=False)
        .agg(subset_count=("image_id", "count"))
    )
    summary = full_grouped.merge(subset_grouped, on="controlled_gt_count_group", how="outer").fillna(0)
    summary["dataset_name"] = dataset_name
    summary["mode_label"] = mode_label
    summary["full_count"] = summary["full_count"].astype(int)
    summary["subset_count"] = summary["subset_count"].astype(int)
    summary["full_total"] = int(len(full_df))
    summary["subset_total"] = int(len(subset_df))
    summary["target_count"] = summary["controlled_gt_count_group"].map(CONTROLLED_SUBSET_TARGETS).fillna(0).astype(int)
    summary["full_ratio"] = summary["full_count"] / max(1, len(full_df))
    summary["subset_ratio"] = summary["subset_count"] / max(1, len(subset_df))
    return summary[[
        "dataset_name", "mode_label", "controlled_gt_count_group", "full_count", "target_count",
        "subset_count", "full_total", "subset_total", "full_ratio", "subset_ratio",
    ]]

def get_image_records(dataset_name, smoke=False):
    df = valid_records_by_dataset[dataset_name].copy()
    df = df[df["readable"] == True].reset_index(drop=True)

    if smoke:
        return select_smoke_records(df)

    if USE_LARGE_VALID_SUBSET_FOR_TUNING:
        if dataset_name in LARGE_VALID_SUBSET_CACHE:
            return LARGE_VALID_SUBSET_CACHE[dataset_name].copy().reset_index(drop=True)

        mode_label = get_run_mode_label(smoke=False)
        selected = None
        selected_source = ""

        if USE_EXISTING_LARGE_VALID_SUBSET_SELECTED_IMAGES:
            selected, existing_path = load_existing_selected_images_for_dataset(dataset_name, df)
            if selected is not None:
                selected_source = relative_path(existing_path)

        if selected is None and USE_CONTROLLED_EQUAL_SUBSET_FOR_TUNING:
            selected, availability_df = select_controlled_equal_subset(
                df=df,
                targets=CONTROLLED_SUBSET_TARGETS,
                random_state=CONTROLLED_SUBSET_RANDOM_STATE,
            )
            selected_source = "controlled_equal_gt_count_sampling"
            save_csv(
                availability_df.assign(dataset_name=dataset_name, mode_label=mode_label),
                TABLES_DIR / f"{mode_label}__{dataset_name}__controlled_subset_availability.csv",
                f"{mode_label} controlled subset availability for {dataset_name}",
            )
            controlled_summary = summarize_controlled_subset_distribution(
                full_df=df,
                subset_df=selected,
                dataset_name=dataset_name,
                mode_label=mode_label,
            )
            save_csv(
                controlled_summary,
                TABLES_DIR / f"{mode_label}__{dataset_name}__controlled_subset_distribution.csv",
                f"{mode_label} controlled gt-count distribution for {dataset_name}",
            )

        if selected is None:
            selected = select_proportional_stratified_subset(
                df=df,
                target_count=LARGE_VALID_SUBSET_TARGET_PER_DATASET,
                random_state=LARGE_VALID_SUBSET_RANDOM_STATE,
            )
            selected_source = "proportional_stratified_fallback"

        summary = summarize_subset_distribution(
            full_df=df,
            subset_df=selected,
            dataset_name=dataset_name,
            mode_label=mode_label,
        )
        summary["selected_source"] = selected_source
        save_csv(
            summary,
            TABLES_DIR / f"{mode_label}__{dataset_name}__subset_distribution.csv",
            f"{mode_label} subset gt_count_group distribution for {dataset_name}",
        )

        selected_export = selected.drop(columns=["gt_boxes"], errors="ignore").copy()
        selected_export["selected_source"] = selected_source
        save_csv(
            selected_export,
            TABLES_DIR / f"{mode_label}__{dataset_name}__selected_images.csv",
            f"{mode_label} selected valid images for {dataset_name}",
        )

        LARGE_VALID_SUBSET_CACHE[dataset_name] = selected.reset_index(drop=True)
        return selected.reset_index(drop=True)

    return df

def compute_raw_scored_windows_for_model_stride(model, model_row, image_records, threshold_column, stride_divisor):
    """Compute raw scored windows once for one model and one stride.

    This is the V5 optimization. It is not a disk cache and it is not shared
    across models. The returned list is kept only while the current model is
    being tuned, then released.
    """
    raw_outputs = []
    dataset_name = model_row["dataset_name"]
    patch_set = model_row["patch_set"]
    patch_size = int(model_row["patch_size"])
    feature_set = model_row["feature_set"]

    for image_idx, (_, image_record) in enumerate(image_records.iterrows(), start=1):
        start_time = time.time()
        image_bgr = cv2.imread(str(image_record["image_path"]))
        if image_bgr is None:
            raise ValueError(f"Could not read image: {image_record['image_path']}")

        h, w = image_bgr.shape[:2]
        scored_windows, stride_px = score_image_windows(
            model=model,
            image_bgr=image_bgr,
            dataset_name=dataset_name,
            patch_set=patch_set,
            patch_size=patch_size,
            feature_set=feature_set,
            stride_divisor=int(stride_divisor),
            threshold_column=threshold_column,
        )
        raw_outputs.append({
            "image_id": image_record["image_id"],
            "gt_boxes": image_record["gt_boxes"],
            "scored_windows": scored_windows,
            "scored_window_count": len(scored_windows),
            "window_count": len(scored_windows),
            "stride_px": stride_px,
            "image_width": w,
            "image_height": h,
            "raw_scoring_runtime_seconds": time.time() - start_time,
        })

    return raw_outputs

def evaluate_param_combo_from_raw_outputs(raw_outputs, model_row, params, phase_name):
    """Evaluate one threshold/post-processing combination using precomputed raw scores."""
    combo_start = time.time()
    image_outputs = []
    patch_size = int(model_row["patch_size"])
    threshold = float(params["threshold_value"])

    for raw in raw_outputs:
        post_start = time.time()
        candidates = build_candidate_detections(raw["scored_windows"], threshold)
        detections = weighted_center_cluster(
            candidates=candidates,
            cluster_iou_threshold=float(params["cluster_iou_threshold"]),
            weighted_box_size_factor=float(params["weighted_box_size_factor"]),
            patch_size=patch_size,
            threshold=threshold,
            image_width=raw["image_width"],
            image_height=raw["image_height"],
        )
        image_outputs.append({
            "image_id": raw["image_id"],
            "gt_boxes": raw["gt_boxes"],
            "detections": detections,
            "scored_window_count": raw["scored_window_count"],
            "window_count": raw["window_count"],
            "candidate_count": len(candidates),
            "final_detection_count": len(detections),
            "stride_px": raw["stride_px"],
            # This is only post-processing time for this combo because raw scoring is reused.
            "runtime_seconds": time.time() - post_start,
        })

    result = {
        "phase": phase_name,
        "dataset_name": model_row["dataset_name"],
        "algorithm_name": model_row["algorithm_name"],
        "model_file_name": model_row["model_file_name"],
        "model_path": model_row["model_path_resolved"],
        "patch_set": model_row["patch_set"],
        "patch_size": int(model_row["patch_size"]),
        "feature_set": model_row["feature_set"],
        "threshold_column": params["threshold_column"],
        "threshold_value": float(params["threshold_value"]),
        "stride_divisor": int(params["stride_divisor"]),
        "cluster_iou_threshold": float(params["cluster_iou_threshold"]),
        "weighted_box_size_factor": float(params["weighted_box_size_factor"]),
        "postprocess_method": POSTPROCESS_METHOD,
        "status": "OK",
        "raw_score_reuse_mode": "per_model_per_stride_in_memory",
        "raw_scoring_total_runtime_seconds": float(sum(o.get("raw_scoring_runtime_seconds", 0.0) for o in raw_outputs)),
    }
    result.update(evaluate_detection_outputs(image_outputs))
    result["postprocess_total_runtime_seconds"] = result.get("total_runtime_seconds", 0.0)
    result["estimated_total_runtime_seconds_with_raw_scoring"] = result["raw_scoring_total_runtime_seconds"] + result["postprocess_total_runtime_seconds"]
    result["combo_wall_runtime_seconds"] = time.time() - combo_start
    return result

def evaluate_param_combo_for_model(model, model_row, image_records, params, phase_name):
    """Compatibility fallback. V5 normally uses raw score reuse instead."""
    outputs = []
    for _, image_record in image_records.iterrows():
        outputs.append(run_detection_for_image(
            model=model,
            image_record=image_record,
            dataset_name=model_row["dataset_name"],
            patch_set=model_row["patch_set"],
            patch_size=int(model_row["patch_size"]),
            feature_set=model_row["feature_set"],
            threshold_column=params["threshold_column"],
            threshold=float(params["threshold_value"]),
            stride_divisor=int(params["stride_divisor"]),
            cluster_iou_threshold=float(params["cluster_iou_threshold"]),
            weighted_box_size_factor=float(params["weighted_box_size_factor"]),
        ))

    result = {
        "phase": phase_name,
        "dataset_name": model_row["dataset_name"],
        "algorithm_name": model_row["algorithm_name"],
        "model_file_name": model_row["model_file_name"],
        "model_path": model_row["model_path_resolved"],
        "patch_set": model_row["patch_set"],
        "patch_size": int(model_row["patch_size"]),
        "feature_set": model_row["feature_set"],
        "threshold_column": params["threshold_column"],
        "threshold_value": float(params["threshold_value"]),
        "stride_divisor": int(params["stride_divisor"]),
        "cluster_iou_threshold": float(params["cluster_iou_threshold"]),
        "weighted_box_size_factor": float(params["weighted_box_size_factor"]),
        "postprocess_method": POSTPROCESS_METHOD,
        "status": "OK",
        "raw_score_reuse_mode": "disabled",
    }
    result.update(evaluate_detection_outputs(outputs, image_records))
    return result

def select_top_thresholds(phase1_model_results, top_n=PHASE2_TOP_THRESHOLD_COUNT):
    df = pd.DataFrame([r for r in phase1_model_results if r.get("status") == "OK"])
    if df.empty:
        return []
    ranked = sort_results(df, primary_metric=DEFAULT_PRIMARY_METRIC, iou_key="0_2")
    values = []
    for value in ranked["threshold_value"].tolist():
        if float(value) not in values:
            values.append(float(value))
        if len(values) >= top_n:
            break
    return values

def run_tuning_group(dataset_name, algorithm_name, smoke=False):
    group_label = f"{dataset_name}__{algorithm_name}"
    mode_label = get_run_mode_label(smoke=smoke)
    print(f"\n=== Running {mode_label}: {group_label} ===")

    phase1_path = TABLES_DIR / f"{mode_label}__{group_label}__phase1_results.csv"
    phase2_path = TABLES_DIR / f"{mode_label}__{group_label}__phase2_results.csv"

    if (not smoke) and phase2_path.exists() and not OVERWRITE_DETECTION_RESULTS:
        print(f"Existing result found and OVERWRITE_DETECTION_RESULTS=False. Loading: {phase2_path}")
        phase1_loaded = pd.read_csv(phase1_path) if phase1_path.exists() else pd.DataFrame()
        return phase1_loaded, pd.read_csv(phase2_path)

    group_models = get_group_models(dataset_name, algorithm_name, smoke=smoke)
    image_records = get_image_records(dataset_name, smoke=smoke)
    baseline = get_baseline_params(dataset_name, algorithm_name)
    threshold_values = get_threshold_values(dataset_name, algorithm_name)

    phase1_results, phase2_results = [], []

    for model_index, (_, model_row) in enumerate(group_models.iterrows(), start=1):
        model_path = Path(model_row["model_path_resolved"])
        print(f"[{model_index}/{len(group_models)}] {model_row['model_file_name']}")
        if not model_path.exists():
            phase1_results.append({"phase": "load_model", "dataset_name": dataset_name, "algorithm_name": algorithm_name, "model_file_name": model_row["model_file_name"], "status": "FAILED", "error_message": f"Missing model: {model_path}"})
            continue

        model = load_model_artifact(model_path)
        raw_outputs_by_stride = {}

        def get_raw_outputs_for_stride(stride_divisor):
            stride_divisor = int(stride_divisor)
            if (not USE_PER_MODEL_STRIDE_RAW_SCORE_REUSE) or (stride_divisor not in raw_outputs_by_stride):
                print(f"  Scoring raw windows once for stride_divisor={stride_divisor}...")
                raw_outputs_by_stride[stride_divisor] = compute_raw_scored_windows_for_model_stride(
                    model=model,
                    model_row=model_row,
                    image_records=image_records,
                    threshold_column=baseline["threshold_column"],
                    stride_divisor=stride_divisor,
                )
                total_windows = sum(o["scored_window_count"] for o in raw_outputs_by_stride[stride_divisor])
                total_seconds = sum(o["raw_scoring_runtime_seconds"] for o in raw_outputs_by_stride[stride_divisor])
                print(f"    raw windows={total_windows:,}, raw scoring seconds={total_seconds:.1f}")
            return raw_outputs_by_stride[stride_divisor]

        this_model_phase1 = []
        baseline_stride = int(STRIDE_DIVISOR_GRID[dataset_name][0])

        for threshold_value in threshold_values:
            params = {
                "threshold_column": baseline["threshold_column"],
                "threshold_value": float(threshold_value),
                "stride_divisor": baseline_stride,
                "cluster_iou_threshold": baseline["cluster_iou_threshold"],
                "weighted_box_size_factor": baseline["weighted_box_size_factor"],
            }
            try:
                if USE_PER_MODEL_STRIDE_RAW_SCORE_REUSE:
                    raw_outputs = get_raw_outputs_for_stride(baseline_stride)
                    result = evaluate_param_combo_from_raw_outputs(raw_outputs, model_row, params, "phase1_threshold_sweep")
                else:
                    result = evaluate_param_combo_for_model(model, model_row, image_records, params, "phase1_threshold_sweep")
                print(f"  P1 thr={threshold_value}: F1@0.20={result.get('detection_f1_iou_0_2', 0):.4f}, F1@0.30={result.get('detection_f1_iou_0_3', 0):.4f}")
            except Exception as exc:
                result = {"phase": "phase1_threshold_sweep", "dataset_name": dataset_name, "algorithm_name": algorithm_name, "model_file_name": model_row["model_file_name"], "threshold_value": threshold_value, "status": "FAILED", "error_message": str(exc), "traceback": traceback.format_exc()}
                print("  ERROR P1:", exc)
            this_model_phase1.append(result)
            phase1_results.append(result)

        top_thresholds = select_top_thresholds(this_model_phase1)

        if smoke and not SMOKE_USE_FULL_PHASE2_GRID:
            # Smoke mode checks that Phase 2 can run; it does not perform the full postprocess grid.
            # This keeps smoke mode short and avoids confusing it with real tuning.
            phase2_thresholds = top_thresholds[:1]
            phase2_stride_grid = [baseline["stride_divisor"]]
            phase2_cluster_grid = [baseline["cluster_iou_threshold"]]
            phase2_box_factor_grid = [baseline["weighted_box_size_factor"]]
            print("  Phase 2 smoke threshold:", phase2_thresholds)
        else:
            phase2_thresholds = top_thresholds
            phase2_stride_grid = STRIDE_DIVISOR_GRID[dataset_name]
            phase2_cluster_grid = CLUSTER_IOU_THRESHOLD_GRID
            phase2_box_factor_grid = WEIGHTED_BOX_SIZE_FACTOR_GRID
            print("  Phase 2 top thresholds:", phase2_thresholds)

        for stride_divisor in phase2_stride_grid:
            try:
                raw_outputs = get_raw_outputs_for_stride(stride_divisor) if USE_PER_MODEL_STRIDE_RAW_SCORE_REUSE else None
            except Exception as exc:
                result = {"phase": "phase2_raw_scoring", "dataset_name": dataset_name, "algorithm_name": algorithm_name, "model_file_name": model_row["model_file_name"], "stride_divisor": stride_divisor, "status": "FAILED", "error_message": str(exc), "traceback": traceback.format_exc()}
                phase2_results.append(result)
                print("  ERROR raw scoring P2:", exc)
                continue

            for threshold_value in phase2_thresholds:
                for cluster_iou_threshold in phase2_cluster_grid:
                    for weighted_box_size_factor in phase2_box_factor_grid:
                        params = {
                            "threshold_column": baseline["threshold_column"],
                            "threshold_value": float(threshold_value),
                            "stride_divisor": int(stride_divisor),
                            "cluster_iou_threshold": float(cluster_iou_threshold),
                            "weighted_box_size_factor": float(weighted_box_size_factor),
                        }
                        try:
                            if USE_PER_MODEL_STRIDE_RAW_SCORE_REUSE:
                                result = evaluate_param_combo_from_raw_outputs(raw_outputs, model_row, params, "phase2_postprocess_sweep")
                            else:
                                result = evaluate_param_combo_for_model(model, model_row, image_records, params, "phase2_postprocess_sweep")
                            print(f"  P2 thr={threshold_value}, stride={stride_divisor}, cluster={cluster_iou_threshold}, box={weighted_box_size_factor}: F1@0.20={result.get('detection_f1_iou_0_2', 0):.4f}")
                        except Exception as exc:
                            result = {"phase": "phase2_postprocess_sweep", "dataset_name": dataset_name, "algorithm_name": algorithm_name, "model_file_name": model_row["model_file_name"], "threshold_value": threshold_value, "stride_divisor": stride_divisor, "cluster_iou_threshold": cluster_iou_threshold, "weighted_box_size_factor": weighted_box_size_factor, "status": "FAILED", "error_message": str(exc), "traceback": traceback.format_exc()}
                            print("  ERROR P2:", exc)
                        phase2_results.append(result)

        del model
        del raw_outputs_by_stride

    phase1_df = pd.DataFrame(phase1_results)
    phase2_df = pd.DataFrame(phase2_results)
    save_csv(phase1_df, phase1_path, f"{mode_label} Phase 1 results for {group_label}")
    save_csv(phase2_df, phase2_path, f"{mode_label} Phase 2 results for {group_label}")
    return phase1_df, phase2_df


## 13. Smoke Mode

Bu bölüm yalnızca kısa pipeline kontrolü için kullanılır; tam tuning çalıştırmaz.


In [ ]:
if RUN_SMOKE_MODE:
    smoke_phase1_all, smoke_phase2_all = [], []
    for dataset_name in ["dataset1_varroa", "dataset2_honeybee"]:
        for algorithm_name in ["linear_svm", "rbf_svm", "random_forest"]:
            p1, p2 = run_tuning_group(dataset_name, algorithm_name, smoke=True)
            smoke_phase1_all.append(p1)
            smoke_phase2_all.append(p2)

    smoke_phase1_df = pd.concat(smoke_phase1_all, ignore_index=True) if smoke_phase1_all else pd.DataFrame()
    smoke_phase2_df = pd.concat(smoke_phase2_all, ignore_index=True) if smoke_phase2_all else pd.DataFrame()

    save_csv(smoke_phase1_df, TABLES_DIR / "smoke_detection_tuning_phase1_results.csv", "Smoke mode Phase 1 results")
    save_csv(smoke_phase2_df, TABLES_DIR / "smoke_detection_tuning_phase2_results.csv", "Smoke mode Phase 2 results")
    display(smoke_phase2_df.head(DISPLAY_PREVIEW_ROWS))
else:
    print("RUN_SMOKE_MODE is False; skipping smoke mode.")


## 14. Dataset1 Linear SVM Refinement

Bu bölüm Dataset1 Linear SVM adaylarını refinement akışına sokar.


In [ ]:
if RUN_FULL_VALID_TUNING:
    phase1_dataset1_varroa_linear_svm, phase2_dataset1_varroa_linear_svm = run_tuning_group("dataset1_varroa", "linear_svm", smoke=False)
    display(sort_results(phase2_dataset1_varroa_linear_svm).head(DISPLAY_PREVIEW_ROWS))
else:
    print("RUN_FULL_VALID_TUNING is False; skipping Dataset1 Linear SVM.")


## 15. Dataset1 RBF SVM Refinement

Bu bölüm Dataset1 RBF SVM adaylarını refinement akışına sokar.


In [ ]:
if RUN_FULL_VALID_TUNING:
    phase1_dataset1_varroa_rbf_svm, phase2_dataset1_varroa_rbf_svm = run_tuning_group("dataset1_varroa", "rbf_svm", smoke=False)
    display(sort_results(phase2_dataset1_varroa_rbf_svm).head(DISPLAY_PREVIEW_ROWS))
else:
    print("RUN_FULL_VALID_TUNING is False; skipping Dataset1 RBF SVM.")


## 16. Dataset1 Random Forest Refinement

Bu bölüm Dataset1 Random Forest adaylarını refinement akışına sokar.


In [ ]:
if RUN_FULL_VALID_TUNING:
    phase1_dataset1_varroa_random_forest, phase2_dataset1_varroa_random_forest = run_tuning_group("dataset1_varroa", "random_forest", smoke=False)
    display(sort_results(phase2_dataset1_varroa_random_forest).head(DISPLAY_PREVIEW_ROWS))
else:
    print("RUN_FULL_VALID_TUNING is False; skipping Dataset1 Random Forest.")


## 17. Dataset2 Linear SVM Refinement

Bu bölüm Dataset2 Linear SVM adaylarını refinement akışına sokar.


In [ ]:
if RUN_FULL_VALID_TUNING:
    phase1_dataset2_honeybee_linear_svm, phase2_dataset2_honeybee_linear_svm = run_tuning_group("dataset2_honeybee", "linear_svm", smoke=False)
    display(sort_results(phase2_dataset2_honeybee_linear_svm).head(DISPLAY_PREVIEW_ROWS))
else:
    print("RUN_FULL_VALID_TUNING is False; skipping Dataset2 Linear SVM.")


## 18. Dataset2 RBF SVM Refinement

Bu bölüm Dataset2 RBF SVM adaylarını refinement akışına sokar.


In [ ]:
if RUN_FULL_VALID_TUNING:
    phase1_dataset2_honeybee_rbf_svm, phase2_dataset2_honeybee_rbf_svm = run_tuning_group("dataset2_honeybee", "rbf_svm", smoke=False)
    display(sort_results(phase2_dataset2_honeybee_rbf_svm).head(DISPLAY_PREVIEW_ROWS))
else:
    print("RUN_FULL_VALID_TUNING is False; skipping Dataset2 RBF SVM.")


## 19. Dataset2 Random Forest Refinement

Bu bölüm Dataset2 Random Forest adaylarını refinement akışına sokar.


In [ ]:
if RUN_FULL_VALID_TUNING:
    phase1_dataset2_honeybee_random_forest, phase2_dataset2_honeybee_random_forest = run_tuning_group("dataset2_honeybee", "random_forest", smoke=False)
    display(sort_results(phase2_dataset2_honeybee_random_forest).head(DISPLAY_PREVIEW_ROWS))
else:
    print("RUN_FULL_VALID_TUNING is False; skipping Dataset2 Random Forest.")


## 20. Sonuç Birleştirme ve Sıralama

Bu bölüm refinement çıktılarının mevcut grup sonuçlarını birleştirir ve sıralama tabloları üretir.


In [ ]:
def load_existing_group_results(phase, mode_label=None):
    if mode_label is None:
        mode_label = get_run_mode_label(smoke=False)
    frames = []
    for dataset_name in ["dataset1_varroa", "dataset2_honeybee"]:
        for algorithm_name in ["linear_svm", "rbf_svm", "random_forest"]:
            path = TABLES_DIR / f"{mode_label}__{dataset_name}__{algorithm_name}__{phase}_results.csv"
            if path.exists():
                frames.append(pd.read_csv(path))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

COMBINED_MODE_LABEL = get_run_mode_label(smoke=False)
tuning_phase1_df = load_existing_group_results("phase1", mode_label=COMBINED_MODE_LABEL)
tuning_phase2_df = load_existing_group_results("phase2", mode_label=COMBINED_MODE_LABEL)

# Backward-compatible aliases for later cells.
full_phase1_df = tuning_phase1_df
full_phase2_df = tuning_phase2_df

if tuning_phase1_df.empty and tuning_phase2_df.empty:
    print(f"No {COMBINED_MODE_LABEL} group result files found yet. Run the six tuning cells first.")
else:
    if not tuning_phase1_df.empty:
        save_csv(
            tuning_phase1_df,
            TABLES_DIR / f"{COMBINED_MODE_LABEL}__detection_tuning_phase1_results.csv",
            f"Combined {COMBINED_MODE_LABEL} Phase 1 tuning results",
        )

    if not tuning_phase2_df.empty:
        save_csv(
            tuning_phase2_df,
            TABLES_DIR / f"{COMBINED_MODE_LABEL}__detection_tuning_phase2_results.csv",
            f"Combined {COMBINED_MODE_LABEL} Phase 2 tuning results",
        )
        save_csv(
            tuning_phase2_df,
            TABLES_DIR / f"{COMBINED_MODE_LABEL}__detection_tuning_results.csv",
            f"Combined {COMBINED_MODE_LABEL} Stage 09 tuning results",
        )

        ranked_iou02 = sort_results(tuning_phase2_df, primary_metric="detection_f1_iou_0_2", iou_key="0_2")
        ranked_iou03 = sort_results(tuning_phase2_df, primary_metric="detection_f1_iou_0_3", iou_key="0_3")
        ranked_iou02.insert(0, "rank_iou_0_2", np.arange(1, len(ranked_iou02) + 1))
        ranked_iou03.insert(0, "rank_iou_0_3", np.arange(1, len(ranked_iou03) + 1))

        save_csv(
            ranked_iou02,
            TABLES_DIR / f"{COMBINED_MODE_LABEL}__ranked_detection_tuning_results_by_iou_0_2.csv",
            f"{COMBINED_MODE_LABEL} ranking by F1@IoU0.20",
        )
        save_csv(
            ranked_iou03,
            TABLES_DIR / f"{COMBINED_MODE_LABEL}__ranked_detection_tuning_results_by_iou_0_3.csv",
            f"{COMBINED_MODE_LABEL} ranking by F1@IoU0.30",
        )

        # Current-mode convenience copies. These are overwritten by the latest combined mode.
        save_csv(ranked_iou02, TABLES_DIR / "ranked_detection_tuning_results_by_iou_0_2.csv", "Current-mode ranking by F1@IoU0.20")
        save_csv(ranked_iou03, TABLES_DIR / "ranked_detection_tuning_results_by_iou_0_3.csv", "Current-mode ranking by F1@IoU0.30")

        comparison_keys = ["model_file_name", "threshold_value", "stride_divisor", "cluster_iou_threshold", "weighted_box_size_factor"]
        comp02_cols = ["rank_iou_0_2", "dataset_name", "algorithm_name", "patch_set", "feature_set"] + comparison_keys + [
            "detection_f1_iou_0_2", "false_positives_per_image_iou_0_2", "detection_recall_iou_0_2",
            "detection_f1_iou_0_3", "false_positives_per_image_iou_0_3", "detection_recall_iou_0_3",
        ]
        comp02 = ranked_iou02[[c for c in comp02_cols if c in ranked_iou02.columns]].copy()
        comp03 = ranked_iou03[["rank_iou_0_3"] + comparison_keys].copy()
        ranking_comparison = comp02.merge(comp03, on=comparison_keys, how="left")
        save_csv(
            ranking_comparison,
            TABLES_DIR / f"{COMBINED_MODE_LABEL}__ranking_comparison_iou_0_2_vs_iou_0_3.csv",
            f"{COMBINED_MODE_LABEL} IoU@0.20 vs IoU@0.30 ranking comparison",
        )
        save_csv(ranking_comparison, TABLES_DIR / "ranking_comparison_iou_0_2_vs_iou_0_3.csv", "Current-mode IoU@0.20 vs IoU@0.30 ranking comparison")

        display(ranked_iou02.head(DISPLAY_PREVIEW_ROWS))


## 21. Sınır Kontrolü

Bu bölüm seçilen parametrelerin grid sınırlarında kalıp kalmadığını kontrol eder.


In [ ]:
def make_boundary_check(results_df):
    if results_df.empty:
        return pd.DataFrame()
    ranked = sort_results(results_df, primary_metric="detection_f1_iou_0_2", iou_key="0_2")
    best_rows = ranked.groupby(["dataset_name", "algorithm_name", "model_file_name"], as_index=False).head(1).copy()
    checks = []
    for _, row in best_rows.iterrows():
        dataset_name = row["dataset_name"]
        algorithm_name = row["algorithm_name"]
        grids = {
            "threshold_value": get_threshold_values(dataset_name, algorithm_name),
            "stride_divisor": STRIDE_DIVISOR_GRID[dataset_name],
            "cluster_iou_threshold": CLUSTER_IOU_THRESHOLD_GRID,
            "weighted_box_size_factor": WEIGHTED_BOX_SIZE_FACTOR_GRID,
        }
        for param_name, values in grids.items():
            values_sorted = sorted(values)
            value = float(row[param_name])
            is_lower = value == float(values_sorted[0])
            is_upper = value == float(values_sorted[-1])
            note = "inside_grid"
            if is_lower:
                note = "lower_boundary_possible_lower_value_needed"
            if is_upper:
                note = "upper_boundary_possible_higher_value_needed"
            checks.append({
                "dataset_name": dataset_name,
                "algorithm_name": algorithm_name,
                "model_file_name": row["model_file_name"],
                "primary_metric": "detection_f1_iou_0_2",
                "primary_metric_value": row.get("detection_f1_iou_0_2", np.nan),
                "parameter_name": param_name,
                "best_value": value,
                "grid_values": json.dumps(values_sorted),
                "is_lower_boundary": is_lower,
                "is_upper_boundary": is_upper,
                "boundary_note": note,
            })
    return pd.DataFrame(checks)

if "full_phase2_df" not in globals() or full_phase2_df.empty:
    print("No full-valid Phase 2 results found. Boundary check will be created after full tuning.")
else:
    boundary_check_df = make_boundary_check(full_phase2_df)
    save_csv(boundary_check_df, TABLES_DIR / f"{get_run_mode_label(smoke=False)}__tuning_boundary_check.csv", f"{get_run_mode_label(smoke=False)} Stage 09 tuning boundary check")
    display(boundary_check_df.head(DISPLAY_PREVIEW_ROWS))


## 22. Nitel Görseller

Bu bölüm isteğe bağlı olarak seçilen modeller için örnek detection görselleri üretir.


In [ ]:
def draw_boxes_on_image(image_bgr, gt_boxes, detections, title):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(image_rgb)
    ax.set_title(title)
    ax.axis("off")

    for gt in gt_boxes:
        ax.add_patch(plt.Rectangle(
            (gt["x1"], gt["y1"]),
            gt["x2"] - gt["x1"],
            gt["y2"] - gt["y1"],
            edgecolor="green",
            fill=False,
            linewidth=2,
        ))
    for det in detections:
        ax.add_patch(plt.Rectangle(
            (det["x1"], det["y1"]),
            det["x2"] - det["x1"],
            det["y2"] - det["y1"],
            edgecolor="red",
            fill=False,
            linewidth=1.5,
            linestyle="--",
        ))
        ax.text(det["x1"], det["y1"], f"{det['score']:.2f}", fontsize=8, color="red")
    return fig

def create_qualitative_examples_for_dataset(best_row, dataset_name):
    model_path = best_row.get("model_path_resolved", best_row.get("model_path"))
    model = load_model_artifact(model_path)
    records = get_image_records(dataset_name, smoke=False)
    selected = pd.concat([
        records[records["gt_count"] > 0].head(4),
        records[records["gt_count"] == 0].head(2),
    ], ignore_index=True).head(QUALITATIVE_EXAMPLES_PER_DATASET)

    saved = []
    for idx, (_, image_record) in enumerate(selected.iterrows(), start=1):
        image_bgr = cv2.imread(str(image_record["image_path"]))
        if image_bgr is None:
            continue
        output = run_detection_for_image(
            model=model,
            image_record=image_record,
            dataset_name=dataset_name,
            patch_set=best_row["patch_set"],
            patch_size=int(best_row["patch_size"]),
            feature_set=best_row["feature_set"],
            threshold_column=best_row["threshold_column"],
            threshold=float(best_row["threshold_value"]),
            stride_divisor=int(best_row["stride_divisor"]),
            cluster_iou_threshold=float(best_row["cluster_iou_threshold"]),
            weighted_box_size_factor=float(best_row["weighted_box_size_factor"]),
        )
        fig = draw_boxes_on_image(image_bgr, image_record["gt_boxes"], output["detections"], f"{dataset_name} | {best_row['model_file_name']} | {image_record['image_id']}")
        out_path = FIGURES_DIR / f"qualitative_{dataset_name}_{idx:02d}.png"
        save_current_figure(out_path, f"Qualitative detection example {idx} for {dataset_name}")
        plt.close(fig)
        saved.append(out_path)
    return saved

if RUN_QUALITATIVE_FIGURES:
    mode_label = get_run_mode_label(smoke=False)
    ranked_path = TABLES_DIR / f"{mode_label}__ranked_detection_tuning_results_by_iou_0_2.csv"
    if not ranked_path.exists():
        ranked_path = TABLES_DIR / "ranked_detection_tuning_results_by_iou_0_2.csv"
    if not ranked_path.exists():
        print("Ranking file not found. Run tuning and ranking before qualitative figures.")
    else:
        ranked = pd.read_csv(ranked_path)
        for dataset_name in ["dataset1_varroa", "dataset2_honeybee"]:
            best = ranked[ranked["dataset_name"] == dataset_name].head(1)
            if not best.empty:
                saved = create_qualitative_examples_for_dataset(best.iloc[0].to_dict(), dataset_name)
                print(f"Saved {len(saved)} qualitative examples for {dataset_name}")
else:
    print("RUN_QUALITATIVE_FIGURES is False; skipping qualitative figures.")


## 23. Final Durum

Bu bölüm threshold refinement durum özetini tablo olarak kaydeder.


In [ ]:
status_rows = [
    {"item": "stage_name", "value": STAGE_NAME},
    {"item": "notebook_name", "value": NOTEBOOK_NAME},
    {"item": "candidate_model_count", "value": len(candidate_models_df)},
    {"item": "postprocess_method", "value": POSTPROCESS_METHOD},
    {"item": "default_primary_metric", "value": DEFAULT_PRIMARY_METRIC},
    {"item": "strict_comparison_metric", "value": STRICT_COMPARISON_METRIC},
    {"item": "run_smoke_mode", "value": RUN_SMOKE_MODE},
    {"item": "run_full_valid_tuning", "value": RUN_FULL_VALID_TUNING},
    {"item": "smoke_use_full_phase2_grid", "value": SMOKE_USE_FULL_PHASE2_GRID},
    {"item": "use_per_model_stride_raw_score_reuse", "value": USE_PER_MODEL_STRIDE_RAW_SCORE_REUSE},
    {"item": "use_large_valid_subset_for_tuning", "value": USE_LARGE_VALID_SUBSET_FOR_TUNING},
    {"item": "configured_tuning_mode_label", "value": get_run_mode_label(smoke=False)},
    {"item": "stage10_selection_policy", "value": "results_review_after_stage09"},
]

if "LARGE_VALID_SUBSET_TARGET_PER_DATASET" in globals():
    status_rows.append({"item": "large_valid_subset_target_per_dataset", "value": LARGE_VALID_SUBSET_TARGET_PER_DATASET})
if "LARGE_VALID_SUBSET_STRATEGY" in globals():
    status_rows.append({"item": "large_valid_subset_strategy", "value": LARGE_VALID_SUBSET_STRATEGY})
if "USE_EXISTING_LARGE_VALID_SUBSET_SELECTED_IMAGES" in globals():
    status_rows.append({"item": "use_existing_large_valid_subset_selected_images", "value": USE_EXISTING_LARGE_VALID_SUBSET_SELECTED_IMAGES})
if "USE_CONTROLLED_EQUAL_SUBSET_FOR_TUNING" in globals():
    status_rows.append({"item": "use_controlled_equal_subset_for_tuning", "value": USE_CONTROLLED_EQUAL_SUBSET_FOR_TUNING})

notebook_status_df = pd.DataFrame(status_rows)
save_csv(notebook_status_df, TABLES_DIR / "notebook_status.csv", "Stage 09 notebook status")

display(notebook_status_df)
print("[STATUS] 02_detection_tuning_threshold_refinement_large_subset tamamlandı veya mevcut sonuçlar üzerinden özetlendi.")
